# 03-frcnn-train-2_fully_commented

Fully commented version. Comments explain the purpose of each important line/block and note Windows/Linux/macOS path considerations.


In [ ]:
# 03-frcnn-train-2_fully_commented: FULLY COMMENTED VERSION
# Every important code line/block includes a comment explaining what it does and why it is used.
# For Windows/Linux/macOS users, replace Kaggle-only paths such as /kaggle/working with your local project folder.



# ============================================================================
# # Faster R-CNN
# 
# ## Programmer Notes
# 
# This notebook now includes comments for Kaggle and local Windows/Linux/macOS use. The code remains Kaggle-compatible, but local users must replace Kaggle paths and secrets with local folders/environment variables.
# ============================================================================


## Cell 1
Read the comments inside the code cell for line-by-line purpose and workflow explanation.


In [ ]:
# -----------------------------------------------------------------------------
# CROSS-PLATFORM NOTES FOR WINDOWS / LINUX / macOS
# -----------------------------------------------------------------------------
# This notebook was originally written for Kaggle. Kaggle uses Linux paths such
# as /kaggle/working and /kaggle/input. Those paths will NOT exist on Windows,
# macOS, or a normal Linux computer unless you create them manually.
#
# For local Windows/Linux/macOS use:
#   1. Create a project folder, for example:
#        Windows: C:\Users\YourName\CPE
#        macOS:   /Users/YourName/CPE
#        Linux:   /home/YourName/CPE
#   2. Replace Kaggle-only paths like /kaggle/working with your project folder.
#   3. Prefer os.path.join(...) or pathlib.Path(...) instead of typing slashes.
#      Python will automatically handle Windows backslashes and macOS/Linux
#      forward slashes.
#   4. GPU training needs an NVIDIA GPU + CUDA-compatible PyTorch. On Kaggle,
#      enable GPU in Notebook Settings. On local machines, install the correct
#      PyTorch build from the official PyTorch selector.
# -----------------------------------------------------------------------------
# CELL PURPOSE:
# Installs/imports Faster R-CNN requirements and confirms CUDA GPU is enabled.
# Kaggle users: T4/T4x2 is recommended. Local users need NVIDIA CUDA setup.

# Purpose: Imports a Python module/library needed by the notebook.
import subprocess, sys, warnings
# Purpose: Hides non-critical warning messages so notebook output stays readable.
warnings.filterwarnings("ignore")
# Purpose: Loops through a list/collection and repeats the indented code for each item.
for pkg in ["pycocotools", "matplotlib"]:
    # Purpose: Starts a safe block for code that might fail on some systems.
    try:
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        __import__(pkg)
    # Purpose: Handles an expected error so the notebook can continue or show a clearer message.
    except ImportError:
        # Purpose: Runs a pip command from inside Python to install missing packages automatically.
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
# Purpose: Imports a Python module/library needed by the notebook.
import torch
# Purpose: Stops the notebook when no CUDA GPU is detected, because training will be too slow or unsupported.
if not torch.cuda.is_available():
    # Purpose: Stops execution with a clear error when a required condition is missing.
    raise RuntimeError("Enable GPU T4 x2")
# Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
print("CUDA OK", torch.__version__)


## Cell 2
Read the comments inside the code cell for line-by-line purpose and workflow explanation.


In [ ]:
# CELL PURPOSE:
# Loads config.json, resolves dataset paths, sets seeds, chooses CPU/GPU device,
# and prepares results.csv logging. For Windows/Linux/macOS, update the config
# path and dataset folder locations before running locally.

# Purpose: Imports a Python module/library needed by the notebook.
import json, os, csv, random
# Purpose: Imports specific classes/functions so the code can use them directly.
from datetime import datetime
# Purpose: Imports a Python module/library needed by the notebook.
import numpy as np
# Purpose: Imports a Python module/library needed by the notebook.
import matplotlib.pyplot as plt
# Purpose: Imports a Python module/library needed by the notebook.
import torch
# Purpose: Imports a Python module/library needed by the notebook.
import torch.optim as optim
# Purpose: Imports specific classes/functions so the code can use them directly.
from PIL import Image
# Purpose: Imports specific classes/functions so the code can use them directly.
from torch.utils.data import Dataset, DataLoader
# Purpose: Imports specific classes/functions so the code can use them directly.
from torchvision import transforms
# Purpose: Imports specific classes/functions so the code can use them directly.
from torchvision.models.detection import (fasterrcnn_resnet50_fpn_v2,FasterRCNN_ResNet50_FPN_V2_Weights,)
# Purpose: Imports specific classes/functions so the code can use them directly.
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


# Purpose: Creates or updates a variable used later in the notebook workflow.
KAGGLE_INPUT_ROOT = "/kaggle/input/datasets/nikachuu/data-prep"
# Purpose: Creates or updates a variable used later in the notebook workflow.
WORKING_CONFIG = "/kaggle/working/config.json"
# Purpose: Creates or updates a variable used later in the notebook workflow.
INPUT_CONFIG = os.path.join(KAGGLE_INPUT_ROOT, "config.json")

# Purpose: Defines a reusable function so the same logic can be called multiple times.
def load_config():
    # Purpose: Runs the indented code only when the condition is true.
    if os.path.isfile(WORKING_CONFIG):
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        config_path = WORKING_CONFIG
    # Purpose: Checks another condition if the previous if condition was false.
    elif os.path.isfile(INPUT_CONFIG):
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        config_path = INPUT_CONFIG
    # Purpose: Runs this block when none of the previous conditions matched.
    else:
        # Purpose: Stops execution with a clear error when a required condition is missing.
        raise FileNotFoundError
    # Purpose: Opens a file safely and automatically closes it after the block finishes.
    with open(config_path) as f:
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        config = json.load(f)
    # Purpose: Starts a multi-line Python structure/call; the following indented lines provide its values.
    roots = [
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        config.get("dataset_path"),
        # Purpose: Builds a file path in a way that works on Windows, Linux, and macOS.
        os.path.join(KAGGLE_INPUT_ROOT, "dataset"),
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        "/kaggle/working/dataset",]
    # Purpose: Loops through a list/collection and repeats the indented code for each item.
    for root in roots:
        # Purpose: Runs the indented code only when the condition is true.
        if not root:
            # Purpose: Skips the rest of the current loop iteration and moves to the next item.
            continue
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        train_ann = os.path.join(root, "train", "_annotations.coco.json")
        # Purpose: Runs the indented code only when the condition is true.
        if os.path.isfile(train_ann):
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            config["dataset_path"] = root
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            config["train_ann"] = train_ann
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            config["valid_ann"] = os.path.join(root, "valid", "_annotations.coco.json")
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            config["test_ann"] = os.path.join(root, "test", "_annotations.coco.json")
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            config["train_img_dir"] = os.path.join(root, "train")
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            config["valid_img_dir"] = os.path.join(root, "valid")
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            config["test_img_dir"] = os.path.join(root, "test")
            # Purpose: Exits the current loop early because the needed condition was already found.
            break
    # Purpose: Runs this block when none of the previous conditions matched.
    else:
        # Purpose: Stops execution with a clear error when a required condition is missing.
        raise FileNotFoundError
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    config["save_dir"] = config.get("save_dir") or "/kaggle/working/results"
    # Purpose: Creates the output folder if it does not already exist.
    os.makedirs(config["save_dir"], exist_ok=True)
    # Purpose: Sends the computed result back to the function caller.
    return config, config_path

# Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
config, CONFIG_PATH = load_config()

# Purpose: Sets the experiment name used in saved folders, plots, and results logs.
Experiment_Name = config["Experiment_Name"]
# Purpose: Selects which Roboflow dataset version to download/use.
dataset_version = config["dataset_version"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
Confidence_Threshold = config["Confidence_Threshold"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
epochs_rcnn = config["epochs_rcnn"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
lr_rcnn = config["lr_rcnn"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
PATIENCE = config["PATIENCE"]
# Purpose: Sets the fixed seed value used for reproducible splits and training behavior.
SEED = config["SEED"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
save_dir = config["save_dir"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
train_ann = config["train_ann"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
valid_ann = config["valid_ann"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
test_ann = config["test_ann"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
train_img_dir = config["train_img_dir"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
valid_img_dir = config["valid_img_dir"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
test_img_dir = config["test_img_dir"]
# Purpose: Creates or updates a variable used later in the notebook workflow.
log_file = os.path.join(save_dir, "results.csv")

# Purpose: Fixes Python random behavior so experiments are more reproducible.
random.seed(SEED)
# Purpose: Fixes NumPy random behavior so data splits and metrics are reproducible.
np.random.seed(SEED)
# Purpose: Fixes PyTorch random behavior for more reproducible training.
torch.manual_seed(SEED)
# Purpose: Fixes PyTorch CUDA random behavior when using one or more GPUs.
torch.cuda.manual_seed_all(SEED)
# Purpose: Chooses GPU when available, otherwise falls back to CPU.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Purpose: Defines a reusable function so the same logic can be called multiple times.
def log_result(exp, model, acc, prec, rec, f1, thresh, variant, ds_ver, notes=""):
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    row = {"timestamp": datetime.now().isoformat(timespec="seconds"),"experiment": exp, "model": model,"accuracy": f"{acc:.2f}", "precision": f"{prec:.2f}","recall": f"{rec:.2f}", "f1": f"{f1:.2f}","threshold": thresh, "variant": variant,"dataset_version": ds_ver, "notes": notes,}
    # Purpose: Creates or updates a variable used later in the notebook workflow.
    write_header = not os.path.exists(log_file)
    # Purpose: Opens a file safely and automatically closes it after the block finishes.
    with open(log_file, "a", newline="") as f:
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        w = csv.DictWriter(f, fieldnames=row.keys())
        # Purpose: Runs the indented code only when the condition is true.
        if write_header:
            # Purpose: Writes a row/header into the CSV results file.
            w.writeheader()
        # Purpose: Writes a row/header into the CSV results file.
        w.writerow(row)
    # Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
    print("Logged to", log_file)

# Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
print("Device:", DEVICE)


## Cell 3
Read the comments inside the code cell for line-by-line purpose and workflow explanation.


In [ ]:
# CELL PURPOSE:
# Builds a PyTorch Dataset/DataLoader from COCO annotations and initializes
# Faster R-CNN with the correct number of classes.

# Purpose: Creates a custom PyTorch Dataset for COCO object detection data.
class RhizomeDetectionDataset(Dataset):
    # Purpose: Defines a reusable function so the same logic can be called multiple times.
    def __init__(self, ann_file, img_dir, augment=False):
        # Purpose: Opens a file safely and automatically closes it after the block finishes.
        with open(ann_file) as f:
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            coco = json.load(f)
        # Purpose: Stores the folder where image files are located.
        self.img_dir = img_dir
        # Purpose: Stores the image metadata list from the COCO annotation file.
        self.images  = coco["images"]
        # Purpose: Stores annotations grouped by image ID for fast lookup.
        self.ann_map = {}
        # Purpose: Loops through a list/collection and repeats the indented code for each item.
        for ann in coco["annotations"]:
            # Purpose: Stores annotations grouped by image ID for fast lookup.
            self.ann_map.setdefault(ann["image_id"], []).append(ann)
        # Purpose: Stores optional image augmentation used only during training.
        self.aug = transforms.Compose([
            # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
            transforms.RandomHorizontalFlip(p=0.5),
            # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
            transforms.ColorJitter(brightness=0.10, contrast=0.10),
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        ]) if augment else None

    # Purpose: Tells PyTorch how many samples are in this Dataset.
    def __len__(self):
        # Purpose: Sends the computed result back to the function caller.
        return len(self.images)

    # Purpose: Tells PyTorch how to load one image and its target boxes/labels.
    def __getitem__(self, idx):
        # Purpose: Gets metadata for the current image index.
        info = self.images[idx]
        # Purpose: Loads the image file and converts it to RGB.
        img  = Image.open(os.path.join(self.img_dir, info["file_name"])).convert("RGB")
        # Purpose: Runs the indented code only when the condition is true.
        if self.aug:
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            img = self.aug(img)
        # Purpose: Prepares lists that will store bounding boxes and object class IDs.
        boxes, labels = [], []
        # Purpose: Loops through a list/collection and repeats the indented code for each item.
        for ann in self.ann_map.get(info["id"], []):
            # Purpose: Reads the COCO bounding box format: x, y, width, height.
            x, y, w, h = [float(v) for v in ann["bbox"]]
            # Purpose: Runs the indented code only when the condition is true.
            if w > 2 and h > 2:
                # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
                W, H  = img.size
                # Purpose: Clamps the top-left box corner so it stays inside the image.
                x1, y1 = max(0, x), max(0, y)
                # Purpose: Clamps the bottom-right box corner so it stays inside the image.
                x2, y2 = min(W, x + w), min(H, y + h)
                # Purpose: Runs the indented code only when the condition is true.
                if x2 > x1 and y2 > y1:
                    # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
                    boxes.append([x1, y1, x2, y2])
                    # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
                    labels.append(ann["category_id"])
        # Purpose: Converts bounding boxes into a PyTorch tensor expected by Faster R-CNN.
        boxes  = torch.tensor(boxes,  dtype=torch.float32) if boxes  else torch.zeros((0, 4))
        # Purpose: Converts class labels into a PyTorch tensor expected by Faster R-CNN.
        labels = torch.tensor(labels, dtype=torch.int64)   if labels else torch.zeros((0,), dtype=torch.int64)
        # Purpose: Sends the computed result back to the function caller.
        return transforms.ToTensor()(img), {
            # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
            "boxes":    boxes,
            # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
            "labels":   labels,
            # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
            "image_id": torch.tensor([info["id"]]),}

# Purpose: Defines batching logic for detection data because each image can have a different number of boxes.
def collate_fn(batch):
    # Purpose: Sends the computed result back to the function caller.
    return tuple(zip(*batch))


# Purpose: Creates the training detection dataset with augmentation enabled.
train_det = RhizomeDetectionDataset(train_ann, train_img_dir, augment=True)
# Purpose: Creates the validation detection dataset without random augmentation.
valid_det = RhizomeDetectionDataset(valid_ann, valid_img_dir, augment=False)
# Purpose: Creates the test detection dataset without random augmentation.
test_det  = RhizomeDetectionDataset(test_ann,  test_img_dir,  augment=False)

# Purpose: Creates the DataLoader that feeds training batches into Faster R-CNN.
train_det_loader = DataLoader(train_det, batch_size=4, shuffle=True,  collate_fn=collate_fn, num_workers=0)
# Purpose: Creates the DataLoader that feeds validation batches into Faster R-CNN.
valid_det_loader = DataLoader(valid_det, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0)
# Purpose: Creates the DataLoader that feeds test batches into Faster R-CNN.
test_det_loader  = DataLoader(test_det,  batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0)

# Purpose: Opens a file safely and automatically closes it after the block finishes.
with open(train_ann) as f:
    # Purpose: Gets metadata for the current image index.
    info = json.load(f)
# Purpose: Adds 1 for the background class because Faster R-CNN requires background + object classes.
NUM_CLASSES = len(info["categories"]) + 1
# Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
print("Num classes:", NUM_CLASSES)

# Purpose: Loads pretrained Faster R-CNN with a ResNet50-FPN backbone.
rcnn_model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT,trainable_backbone_layers=3,).to(DEVICE)
# Purpose: Reads the number of input features in the model classification head.
in_feat = rcnn_model.roi_heads.box_predictor.cls_score.in_features
# Purpose: Replaces the default prediction head so it matches this dataset class count.
rcnn_model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, NUM_CLASSES)
# Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
print("Model Initiated")


## Cell 4
Read the comments inside the code cell for line-by-line purpose and workflow explanation.


In [ ]:
# CELL PURPOSE:
# Trains Faster R-CNN with early stopping, evaluates detection performance,
# saves model/results/plots, and logs final metrics.

# Purpose: Defines a reusable early stopping helper to stop training when validation stops improving.
class EarlyStopping:
    # Purpose: Defines a reusable function so the same logic can be called multiple times.
    def __init__(self, patience=5, mode="min"):
        # Purpose: Stores data on the object so other methods in the class can use it.
        self.patience, self.mode = patience, mode
        # Purpose: Stores data on the object so other methods in the class can use it.
        self.counter, self.best, self.stop = 0, None, False
    # Purpose: Defines a reusable function so the same logic can be called multiple times.
    def __call__(self, val):
        # Purpose: Runs the indented code only when the condition is true.
        if self.best is None:
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            self.best = val
            # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
            return
        # Purpose: Checks whether the latest validation value is meaningfully better than the best one so far.
        improved = val < self.best - 0.001 if self.mode == "min" else val > self.best + 0.001
        # Purpose: Runs the indented code only when the condition is true.
        if improved:
            # Purpose: Stores data on the object so other methods in the class can use it.
            self.best, self.counter = val, 0
        # Purpose: Runs this block when none of the previous conditions matched.
        else:
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            self.counter += 1
            # Purpose: Runs the indented code only when the condition is true.
            if self.counter >= self.patience:
                # Purpose: Creates or updates a variable used later in the notebook workflow.
                self.stop = True

# Purpose: Groups Faster R-CNN training, evaluation, saving, and plotting into one trainer class.
class FasterRCNNTrainer:
    # Purpose: Defines a reusable function so the same logic can be called multiple times.
    def __init__(self, model, device, lr=0.005, patience=5):
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        self.model = model
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        self.device = device
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        self.save_path = os.path.join(save_dir, "rcnn_best.pth")
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        self.history = {"train_loss": [], "val_loss": []}
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        self.best_val = float("inf")
        # Purpose: Creates the optimizer that updates trainable Faster R-CNN weights.
        self.optimizer = optim.SGD(
            # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
            [p for p in model.parameters() if p.requires_grad],
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            lr=lr, momentum=0.9, weight_decay=0.0005)
        # Purpose: Creates a cosine learning-rate schedule that gradually adjusts learning rate.
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            # Purpose: Creates the optimizer that updates trainable Faster R-CNN weights.
            self.optimizer, T_max=epochs_rcnn, eta_min=1e-6)
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        self.es = EarlyStopping(patience=patience)

    # Purpose: Runs one full training or validation pass through a DataLoader.
    def _epoch(self, loader, train=True):
        # Purpose: Switches the model to training mode or evaluation mode.
        self.model.train() if train else self.model.eval()
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        total, n = 0.0, 0
        # Purpose: Uses gradients only during training and disables them during validation for efficiency.
        ctx = torch.enable_grad() if train else torch.no_grad()
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        with ctx:
            # Purpose: Loops through a list/collection and repeats the indented code for each item.
            for imgs, tgts in loader:
                # Purpose: Moves image tensors to the selected CPU/GPU device.
                imgs = [i.to(self.device) for i in imgs]
                # Purpose: Moves target dictionaries to the selected CPU/GPU device.
                tgts = [{k: v.to(self.device) for k, v in t.items()} for t in tgts]
                # Purpose: Adds all Faster R-CNN loss components into one training loss value.
                loss = sum(self.model(imgs, tgts).values())
                # Purpose: Runs the indented code only when the condition is true.
                if train:
                    # Purpose: Creates the optimizer that updates trainable Faster R-CNN weights.
                    self.optimizer.zero_grad()
                    # Purpose: Computes gradients for model weights using backpropagation.
                    loss.backward()
                    # Purpose: Clips large gradients to make training more stable.
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    # Purpose: Creates the optimizer that updates trainable Faster R-CNN weights.
                    self.optimizer.step()
                # Purpose: Creates or updates a variable used later in the notebook workflow.
                total += loss.item()
                # Purpose: Creates or updates a variable used later in the notebook workflow.
                n += 1
        # Purpose: Sends the computed result back to the function caller.
        return total / n if n else 0.0

    # Purpose: Runs the full Faster R-CNN training loop across multiple epochs.
    def fit(self, train_loader, valid_loader, epochs):
        # Purpose: Loops through a list/collection and repeats the indented code for each item.
        for epoch in range(epochs):
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            tr = self._epoch(train_loader, True)
            # Purpose: Creates or updates a variable used later in the notebook workflow.
            vl = self._epoch(valid_loader, False)
            # Purpose: Creates a cosine learning-rate schedule that gradually adjusts learning rate.
            self.scheduler.step()
            # Purpose: Stores data on the object so other methods in the class can use it.
            self.history["train_loss"].append(tr)
            # Purpose: Stores data on the object so other methods in the class can use it.
            self.history["val_loss"].append(vl)
            # Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
            print("Epoch", epoch + 1, "train", round(tr, 4), "val", round(vl, 4))
            # Purpose: Runs the indented code only when the condition is true.
            if vl < self.best_val:
                # Purpose: Creates or updates a variable used later in the notebook workflow.
                self.best_val = vl
                # Purpose: Saves the current best model weights to disk.
                torch.save(self.model.state_dict(), self.save_path)
                # Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
                print("Saved best val", round(vl, 4))
            # Purpose: Stores data on the object so other methods in the class can use it.
            self.es(vl)
            # Purpose: Runs the indented code only when the condition is true.
            if self.es.stop:
                # Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
                print("Early stop epoch", epoch + 1)
                # Purpose: Exits the current loop early because the needed condition was already found.
                break

    # Purpose: Evaluates Faster R-CNN predictions using precision, recall, F1, and mIoU.
    def evaluate(self, test_loader, conf_thresh=0.5, iou_thresh=0.5):
        # Purpose: Stores data on the object so other methods in the class can use it.
        self.model.load_state_dict(torch.load(self.save_path, map_location=self.device))
        # Purpose: Stores data on the object so other methods in the class can use it.
        self.model.eval()
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        tp = fp = fn = 0
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        iou_list = []
        # Purpose: Defines Intersection over Union calculation for matching predicted and ground-truth boxes.
        def iou_fn(b1, b2):
            # Purpose: Clamps the top-left box corner so it stays inside the image.
            x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
            # Purpose: Clamps the bottom-right box corner so it stays inside the image.
            x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
            # Purpose: Computes the overlap area between two bounding boxes.
            inter = max(0, x2 - x1) * max(0, y2 - y1)
            # Purpose: Computes the combined area covered by two boxes.
            union = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
            # Purpose: Sends the computed result back to the function caller.
            return inter / union if union > 0 else 0
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        with torch.no_grad():
            # Purpose: Loops through a list/collection and repeats the indented code for each item.
            for imgs, tgts in test_loader:
                # Purpose: Moves image tensors to the selected CPU/GPU device.
                imgs = [i.to(self.device) for i in imgs]
                # Purpose: Converts model scores into predicted class IDs.
                preds = self.model(imgs)
                # Purpose: Loops through a list/collection and repeats the indented code for each item.
                for pred, tgt in zip(preds, tgts):
                    # Purpose: Creates or updates a variable used later in the notebook workflow.
                    gt = tgt["boxes"].cpu().numpy()
                    # Purpose: Creates or updates a variable used later in the notebook workflow.
                    pb = pred["boxes"].cpu().numpy()
                    # Purpose: Creates or updates a variable used later in the notebook workflow.
                    ps = pred["scores"].cpu().numpy()
                    # Purpose: Keeps only predicted boxes whose confidence score passes the threshold.
                    pb = pb[ps >= conf_thresh]
                    # Purpose: Tracks which ground-truth boxes have already been matched.
                    matched = set()
                    # Purpose: Loops through a list/collection and repeats the indented code for each item.
                    for box in pb:
                        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
                        best_i, best_j = 0, -1
                        # Purpose: Loops through a list/collection and repeats the indented code for each item.
                        for j, gb in enumerate(gt):
                            # Purpose: Runs the indented code only when the condition is true.
                            if j in matched:
                                # Purpose: Skips the rest of the current loop iteration and moves to the next item.
                                continue
                            # Purpose: Creates or updates a variable used later in the notebook workflow.
                            v = iou_fn(box, gb)
                            # Purpose: Runs the indented code only when the condition is true.
                            if v > best_i:
                                # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
                                best_i, best_j = v, j
                        # Purpose: Runs the indented code only when the condition is true.
                        if best_i >= iou_thresh:
                            # Purpose: Creates or updates a variable used later in the notebook workflow.
                            tp += 1
                            # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
                            matched.add(best_j)
                            # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
                            iou_list.append(best_i)
                        # Purpose: Runs this block when none of the previous conditions matched.
                        else:
                            # Purpose: Creates or updates a variable used later in the notebook workflow.
                            fp += 1
                    # Purpose: Creates or updates a variable used later in the notebook workflow.
                    fn += len(gt) - len(matched)
        # Purpose: Computes precision: how many predicted boxes were correct.
        prec = tp / (tp + fp) if (tp + fp) else 0
        # Purpose: Computes recall: how many ground-truth boxes were found.
        rec = tp / (tp + fn) if (tp + fn) else 0
        # Purpose: Computes F1 score as the balance of precision and recall.
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
        # Purpose: Computes mean IoU from matched boxes.
        miou = float(np.mean(iou_list)) if iou_list else 0
        # Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
        print("Precision", round(prec * 100, 2), "Recall", round(rec * 100, 2),
              # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
              "F1", round(f1 * 100, 2), "mIoU", round(miou * 100, 2))
        # Purpose: Sends the computed result back to the function caller.
        return prec, rec, f1, miou

    # Purpose: Plots Faster R-CNN training/validation loss and overfitting gap.
    def plot(self):
        # Purpose: Creates a Matplotlib figure and axes used for drawing a chart.
        fig, ax = plt.subplots(1, 2, figsize=(10, 4))
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        ax[0].plot(self.history["train_loss"], label="train")
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        ax[0].plot(self.history["val_loss"], label="val")
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        ax[0].legend()
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        ax[0].set_title("R-CNN loss")
        # Purpose: Creates or updates a variable used later in the notebook workflow.
        gap = [v - t for v, t in zip(self.history["val_loss"], self.history["train_loss"])]
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        ax[1].bar(range(len(gap)), gap)
        # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
        ax[1].set_title("Val minus train")
        # Purpose: Saves the generated chart as an image file for reports/thesis documentation.
        plt.savefig(os.path.join(save_dir, "rcnn_" + Experiment_Name + ".png"), dpi=150)
        # Purpose: Displays the chart inside the notebook output.
        plt.show()

# Purpose: Creates the Faster R-CNN trainer using the model, device, learning rate, and patience.
rcnn_trainer = FasterRCNNTrainer(rcnn_model, DEVICE, lr=lr_rcnn, patience=PATIENCE)
# Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
rcnn_trainer.fit(train_det_loader, valid_det_loader, epochs_rcnn)
# Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
rcnn_trainer.plot()
# Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
rp, rr, rf, riou = rcnn_trainer.evaluate(test_det_loader, Confidence_Threshold, 0.5)
# Purpose: Creates or updates a variable used later in the notebook workflow.
config["rcnn_save_path"] = rcnn_trainer.save_path
# Purpose: Opens a file safely and automatically closes it after the block finishes.
with open("/kaggle/working/config.json", "w") as f:
    # Purpose: Writes Python dictionary/list data into a JSON file.
    json.dump(config, f, indent=2)
# Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
log_result(Experiment_Name, "FasterRCNN-R50-FPN-V2", rp*100, rp*100, rr*100, rf*100,
           # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
           Confidence_Threshold, "ResNet50-FPN-V2", dataset_version,
           # Purpose: Purpose: this line performs one step in the current cell workflow and feeds its result to the next lines.
           "mIoU=" + str(round(riou*100, 2)))
# Purpose: Prints progress/output so the programmer can verify what the notebook is doing.
print("Done")
